# Snowflake Stage Practical
## D381_StagePractical

**Create files → create a stage → upload → list → query → download → load a table**

Start with one CSV, one JSON, and one XML example. Then repeat the workflow with real Avro and Parquet files and inspect raw text.

This notebook contains Markdown instructions and copyable SQL, Python, and PowerShell blocks. Run SQL in Snowflake; run local commands on your computer. Nothing executes merely by opening this notebook.

**Prerequisites:** basic SQL, a Snowflake account and authorized role, access to an X-Small warehouse, and a local Snowflake CLI or SnowSQL connection for `PUT`/`GET`. Snowsight upload is an alternative. Querying files and loading tables consume compute; internal staged files consume storage.

Documentation checked on **7 September 2026**. Local samples are supplied; the Snowflake commands have not been executed against a live account.

**Suggested route:** complete sections 1–15 for the core CSV/JSON/XML lab; sections 16 onward extend it.

## 1. What is a stage, and where exactly is it stored?

A stage gives Snowflake a named location for files used in data loading, inspection, and unloading. Separate the **stage object** (configuration and permissions) from the **files** it stores or references.

| Stage type | Where the file bytes live | Reference |
|---|---|---|
| Named internal | Snowflake-managed cloud storage associated with your account | `@STAGE_LAB_DB.PUBLIC.LAB_FILES` |
| User internal | Snowflake-managed storage for the current user's stage | `@~` |
| Table internal | Snowflake-managed storage associated with a table's implicit stage | `@%ORDERS` |
| Named external | Your specified S3 bucket, Azure container, or GCS bucket/prefix | `@MY_EXTERNAL_STAGE` |

An internal stage is not a folder on a virtual warehouse's local disk. Snowflake manages its physical backing location; you address it through a stage URI, not a customer-owned bucket path. Suspending the warehouse leaves staged files intact.

An external stage's `URL` identifies the actual storage location. Creating that stage does not copy the external files into Snowflake.

Sources: [CREATE STAGE](https://docs.snowflake.com/en/sql-reference/sql/create-stage), [Internal stage types](https://docs.snowflake.com/en/user-guide/data-load-local-file-system-create-stage).

## 2. What is stored? Files, parsed rows, and table rows

```text
Your computer                     Snowflake-managed cloud storage
orders.csv -- PUT/upload -------> @LAB_FILES/csv/orders.csv
                                       |
                      SELECT + file format: parse on demand
                                       |
                      COPY INTO: persist rows in a table
```

Stages contain source files: CSV, JSON, XML, Avro, Parquet, text, and potentially unstructured files such as PDFs or images. SQL file readers only understand supported formats; uploading a PDF does not make it a SQL table.

Uploading does not automatically insert table rows. A file format tells the parser how to interpret bytes; it neither stores data nor converts the staged file. Direct stage queries inspect parsed records. A loaded native table stores data in Snowflake's managed table representation.

`PUT`, `SELECT`, and `COPY INTO` are three different operations. Unless a load explicitly purges files, the stage copy remains after loading.

Sources: [Local file loading workflow](https://docs.snowflake.com/en/user-guide/data-load-local-file-system), [COPY INTO table](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table).

## 3. Included sample data and expected results

All five structured-format examples represent these same orders:

| order_id | customer | city | amount |
|---:|---|---|---:|
| 101 | Asha | Chennai | 1250.50 |
| 102 | Ravi | Bengaluru | 800.00 |
| 103 | Meena | Hyderabad | 1499.50 |

Expected record count: **3**. Expected total amount: **3550.00**. JSON also includes a channel and one item per order; XML puts the order ID in an attribute.

Files relative to this notebook:

| File | Purpose |
|---|---|
| [orders.csv](data/stage_practical/csv/orders.csv) | Header and three comma-separated records |
| [orders.json](data/stage_practical/json/orders.json) | Array of three objects |
| [orders.xml](data/stage_practical/xml/orders.xml) | One root containing three order elements |
| [orders.avro](data/stage_practical/avro/orders.avro) | Avro object container file with embedded schema |
| [orders.parquet](data/stage_practical/parquet/orders.parquet) | Binary columnar file |
| [notes.txt](data/stage_practical/raw/notes.txt) | Three raw text lines |
| [orders.avsc](data/stage_practical/avro/orders.avsc) | Human-readable Avro schema; do not upload as order data |

The local root used in commands is `C:/Course/DataEng/D38_SnowFlake/data/stage_practical`. Adapt this path if you move the course.

## 4. Create or recreate the sample files locally

The files are already included. You can edit CSV, JSON, XML, and TXT with a text editor and save as UTF-8. Avro and Parquet must be generated with format-aware writers; changing a filename extension does not convert data.

To regenerate all samples, run these commands in **local PowerShell**, from `C:/Course/DataEng`:

```powershell
# Only needed if these optional binary-format libraries are missing:
python -m pip install pyarrow fastavro

python D38_SnowFlake/data/stage_practical/generate_samples.py
```

Re-running the generator replaces its named sample files. It does not contact Snowflake. CSV, JSON, XML, and TXT generation use Python's standard library; Avro and Parquet use the two optional packages.

The full generator below also shows how the sample records and files are constructed.

Sources: [fastavro writer](https://fastavro.readthedocs.io/en/latest/writer.html), [Arrow Parquet reader/writer](https://arrow.apache.org/docs/python/parquet.html).

## Sample-file generator

```python
"""Recreate the D381 samples locally; never connects to Snowflake.

Run: python D38_SnowFlake/data/stage_practical/generate_samples.py
Optional binary dependencies: python -m pip install pyarrow fastavro
Re-running overwrites only this generator's named sample files.
"""
import csv
import json
from pathlib import Path
import xml.etree.ElementTree as ET

ROOT = Path(__file__).resolve().parent
ROWS = [
    {"order_id": 101, "customer": "Asha", "city": "Chennai", "amount": 1250.50},
    {"order_id": 102, "customer": "Ravi", "city": "Bengaluru", "amount": 800.00},
    {"order_id": 103, "customer": "Meena", "city": "Hyderabad", "amount": 1499.50},
]


def generate():
    for name in ("csv", "json", "xml", "avro", "parquet", "raw"):
        (ROOT / name).mkdir(parents=True, exist_ok=True)
    with (ROOT / "csv/orders.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(ROWS[0]))
        writer.writeheader()
        writer.writerows(ROWS)
    events = [dict(row, channel="web", items=[{"sku": "BOOK", "qty": 1}]) for row in ROWS]
    (ROOT / "json/orders.json").write_text(json.dumps(events, indent=2) + "\n", encoding="utf-8")
    root = ET.Element("orders")
    for row in ROWS:
        order = ET.SubElement(root, "order", {"id": str(row["order_id"])})
        for name in ("customer", "city", "amount"):
            ET.SubElement(order, name).text = str(row[name])
    ET.indent(root, space="  ")
    ET.ElementTree(root).write(ROOT / "xml/orders.xml", encoding="utf-8", xml_declaration=True)
    (ROOT / "raw/notes.txt").write_text("Stage practical sample\nBatch: D381\nExpected orders: 3\n", encoding="utf-8")
    schema = {"type": "record", "name": "Order", "namespace": "training",
              "fields": [{"name": "order_id", "type": "long"},
                         {"name": "customer", "type": "string"},
                         {"name": "city", "type": "string"},
                         {"name": "amount", "type": "double"}]}
    (ROOT / "avro/orders.avsc").write_text(json.dumps(schema, indent=2) + "\n", encoding="utf-8")
    try:
        from fastavro import writer, parse_schema
        import pyarrow as pa
        import pyarrow.parquet as pq
    except ImportError:
        print("CSV, JSON, XML, TXT and Avro schema created. Install pyarrow and fastavro for binaries.")
        return
    with (ROOT / "avro/orders.avro").open("wb") as f:
        writer(f, parse_schema(schema), ROWS, codec="null")
    pq.write_table(pa.Table.from_pylist(ROWS), ROOT / "parquet/orders.parquet", compression="snappy")
    print("Created CSV, JSON, XML, Avro, Parquet, and raw text samples in", ROOT)


if __name__ == "__main__":
    generate()
```

## 5. Create the lab database, warehouse, and internal stage

**Run in a Snowflake SQL worksheet or connected SQL client.** Use your authorized training role. It needs the relevant create privileges; alternatively, use an existing database, schema, and warehouse assigned by your administrator.

```sql
CREATE DATABASE IF NOT EXISTS STAGE_LAB_DB;
USE DATABASE STAGE_LAB_DB;
USE SCHEMA PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS STAGE_LAB_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE STAGE_LAB_WH;

CREATE STAGE IF NOT EXISTS LAB_FILES
  DIRECTORY = (ENABLE = TRUE)
  COMMENT = 'D381 training files organized by format';

SHOW STAGES IN SCHEMA STAGE_LAB_DB.PUBLIC;
DESC STAGE LAB_FILES;
```

There is no `URL` clause, so `LAB_FILES` is internal. The stage deliberately has no default file format because this lab stores several formats. Each query names its parser explicitly.

`IF NOT EXISTS` preserves existing objects and settings. If you reuse these names, verify their definitions. For an existing internal lab stage without a directory table, run `ALTER STAGE LAB_FILES SET DIRECTORY = (ENABLE = TRUE);`.

Sources: [CREATE STAGE](https://docs.snowflake.com/en/sql-reference/sql/create-stage), [Directory table management](https://docs.snowflake.com/en/user-guide/data-load-dirtables-manage).

## 6. Create the file formats

**Snowflake SQL.** These names belong to the database/schema selected above.

```sql
CREATE FILE FORMAT IF NOT EXISTS FF_CSV_ORDERS
  TYPE = CSV
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  EMPTY_FIELD_AS_NULL = TRUE;

CREATE FILE FORMAT IF NOT EXISTS FF_JSON_ORDERS
  TYPE = JSON
  STRIP_OUTER_ARRAY = TRUE;

CREATE FILE FORMAT IF NOT EXISTS FF_XML_ORDERS
  TYPE = XML
  STRIP_OUTER_ELEMENT = TRUE
  DISABLE_AUTO_CONVERT = TRUE;

CREATE FILE FORMAT IF NOT EXISTS FF_AVRO_ORDERS TYPE = AVRO;
CREATE FILE FORMAT IF NOT EXISTS FF_PARQUET_ORDERS TYPE = PARQUET;

CREATE FILE FORMAT IF NOT EXISTS FF_RAW_LINES
  TYPE = CSV
  FIELD_DELIMITER = NONE
  RECORD_DELIMITER = '\n'
  SKIP_HEADER = 0
  FIELD_OPTIONALLY_ENCLOSED_BY = NONE
  ESCAPE_UNENCLOSED_FIELD = NONE;

SHOW FILE FORMATS IN SCHEMA STAGE_LAB_DB.PUBLIC;
```

CSV skips its header. JSON exposes each outer-array item as a record. XML exposes each child of `<orders>` as a document and keeps element text from being automatically converted. The raw reader uses one physical line per record with no field splitting.

If these format objects already exist with other definitions, inspect them with `DESC FILE FORMAT` and correct the settings before proceeding.

Source: [CREATE FILE FORMAT](https://docs.snowflake.com/en/sql-reference/sql/create-file-format).

## 7. Upload CSV, JSON, and XML from your computer

**Run `PUT` through a locally connected Snowflake CLI, SnowSQL, or supported driver.** A browser SQL worksheet cannot read your computer's `C:` drive by executing `PUT`.

```sql
PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/csv/orders.csv'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/csv/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;

PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/json/orders.json'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/json/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;

PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/xml/orders.xml'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/xml/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;
```

The file URI is local; the `@...` path is remote. `AUTO_COMPRESS = FALSE` keeps filenames simple for this lab. With automatic compression enabled, files can appear with a `.gz` suffix. Inspect the transfer result and then run `LIST`.

`OVERWRITE = FALSE` avoids replacing an existing staged file; a repeated transfer may be skipped. After deliberately changing a sample, use `OVERWRITE = TRUE` for that specific file.

Source: [PUT](https://docs.snowflake.com/en/sql-reference/sql/put).

## 8. Upload through Snowsight instead

In Snowsight, navigate to the stage through the database object explorer, select `STAGE_LAB_DB` → `PUBLIC` → `LAB_FILES`, then use its file upload action. Select the supplied local file and specify the matching path/prefix (`csv/`, `json/`, or `xml/`) when the upload dialog offers that field.

UI labels can change. After uploading, use `LIST` to confirm the actual remote name. If a file landed at the root, either use that actual path in the query or upload it under the intended prefix. The later examples assume the prefixes shown in section 7.

Choose upload-to-stage for this exercise; a load-to-table workflow performs an additional operation. Uploading the same files through both methods is unnecessary.

Source: [Staging files using Snowsight](https://docs.snowflake.com/en/user-guide/data-load-local-file-system-stage-ui).

## 9. How to create a directory in a stage

Stage paths use object-name prefixes. There is no SQL `mkdir` step for an internal stage. Uploading to `@LAB_FILES/csv/` places the object under the `csv/` prefix. An empty folder need not exist first.

```text
@LAB_FILES/
    csv/orders.csv
    json/orders.json
    xml/orders.xml
    avro/orders.avro
    parquet/orders.parquet
    raw/notes.txt
```

For a future batch, a destination such as `@LAB_FILES/csv/batch_002/` creates that logical path when files are uploaded. Do not upload a second copy for this lab unless you also adjust the expected counts.

A **directory table** is different: it catalogs file metadata, not folders and not business rows. Refresh it after uploads:

```sql
ALTER STAGE STAGE_LAB_DB.PUBLIC.LAB_FILES REFRESH;
```

Source: [Directory table management](https://docs.snowflake.com/en/user-guide/data-load-dirtables-manage).

## 10. List all files, list a prefix, and inspect metadata

```sql
LIST @STAGE_LAB_DB.PUBLIC.LAB_FILES;
LIST @STAGE_LAB_DB.PUBLIC.LAB_FILES/csv/;
LIST @STAGE_LAB_DB.PUBLIC.LAB_FILES PATTERN = '.*[.]json$';
```

`LIST` returns file information such as name, size, checksum metadata, and modification time; it does not return order records. Patterns are regular expressions. Do not assume every listed checksum is a local file's plain MD5 value.

Source: [LIST](https://docs.snowflake.com/en/sql-reference/sql/list).

```sql
ALTER STAGE STAGE_LAB_DB.PUBLIC.LAB_FILES REFRESH;

SELECT relative_path, size, last_modified, file_url
FROM DIRECTORY(@STAGE_LAB_DB.PUBLIC.LAB_FILES)
ORDER BY relative_path;

SELECT relative_path, size
FROM DIRECTORY(@STAGE_LAB_DB.PUBLIC.LAB_FILES)
WHERE relative_path LIKE 'csv/%';
```

The directory query returns one row per file, not one per order. `FILE_URL` is a file reference subject to Snowflake access controls, not automatically a public download link. After only the core uploads, expect three file entries; after all extensions, expect six.

Source: [Query directory tables](https://docs.snowflake.com/en/user-guide/data-load-dirtables-query).

## 11. How a stage query works

```sql
USE DATABASE STAGE_LAB_DB;
USE SCHEMA PUBLIC;
USE WAREHOUSE STAGE_LAB_WH;

SELECT t.$1, t.$2
FROM @LAB_FILES/csv/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_CSV_ORDERS',
   PATTERN => '.*orders[.]csv$') t;
```

`$1` is the first parsed CSV field; `$2` is the second. For JSON, Avro, Parquet, and XML, `$1` holds the parsed record/document representation. A suffix such as `:customer` selects a field in a semi-structured record.

Use both a path and a filename pattern when selecting an exact sample. A stage path can be a prefix rather than an exact filesystem lookup. Avoid querying the mixed-format stage root with a single parser.

Stage queries are intended for inspection and load preparation. For repeated analytics, load the data into tables.

Source: [Query staged files](https://docs.snowflake.com/en/user-guide/querying-stage).

## 12. CSV example: source file

Save this as `csv/orders.csv` under the sample-data root, or use the included file. Upload it using section 7.

```csv
order_id,customer,city,amount
101,Asha,Chennai,1250.5
102,Ravi,Bengaluru,800.0
103,Meena,Hyderabad,1499.5
```

## CSV: query fields, types, and source metadata

```sql
SELECT t.$1::INTEGER AS order_id,
       t.$2::VARCHAR AS customer,
       t.$3::VARCHAR AS city,
       t.$4::NUMBER(10,2) AS amount,
       METADATA$FILENAME AS source_file,
       METADATA$FILE_ROW_NUMBER AS source_row
FROM @LAB_FILES/csv/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_CSV_ORDERS',
   PATTERN => '.*orders[.]csv$') t
ORDER BY order_id;
```

Expected business columns match the three orders in section 3. The header is not an order. File metadata identifies the original source; it is selected explicitly rather than inferred from table rows.

```sql
SELECT COUNT(*) AS order_count,
       SUM(t.$4::NUMBER(10,2)) AS total_amount
FROM @LAB_FILES/csv/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_CSV_ORDERS',
   PATTERN => '.*orders[.]csv$') t;
```

Expected: `ORDER_COUNT = 3`, `TOTAL_AMOUNT = 3550.00`.

Source: [Staged file metadata](https://docs.snowflake.com/en/user-guide/querying-metadata).

## 13. JSON example: source file

Save this as `json/orders.json` under the sample-data root, or use the included file. Upload it using section 7.

```json
[
  {
    "order_id": 101,
    "customer": "Asha",
    "city": "Chennai",
    "amount": 1250.5,
    "channel": "web",
    "items": [
      {
        "sku": "BOOK",
        "qty": 1
      }
    ]
  },
  {
    "order_id": 102,
    "customer": "Ravi",
    "city": "Bengaluru",
    "amount": 800.0,
    "channel": "web",
    "items": [
      {
        "sku": "BOOK",
        "qty": 1
      }
    ]
  },
  {
    "order_id": 103,
    "customer": "Meena",
    "city": "Hyderabad",
    "amount": 1499.5,
    "channel": "web",
    "items": [
      {
        "sku": "BOOK",
        "qty": 1
      }
    ]
  }
]
```

## JSON: inspect objects and extract fields

```sql
SELECT t.$1 AS parsed_json
FROM @LAB_FILES/json/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_JSON_ORDERS',
   PATTERN => '.*orders[.]json$') t;

SELECT t.$1:order_id::INTEGER AS order_id,
       t.$1:customer::VARCHAR AS customer,
       t.$1:city::VARCHAR AS city,
       t.$1:amount::NUMBER(10,2) AS amount,
       t.$1:channel::VARCHAR AS channel
FROM @LAB_FILES/json/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_JSON_ORDERS',
   PATTERN => '.*orders[.]json$') t
ORDER BY order_id;
```

Expected: three orders, each with channel `web`. Field names are case-sensitive. `STRIP_OUTER_ARRAY = TRUE` gives three objects instead of one outer-array value. The parsed representation is not the original file's exact spacing or bytes.

Source: [Query staged files](https://docs.snowflake.com/en/user-guide/querying-stage).

To expand the nested item array:

```sql
SELECT t.$1:order_id::INTEGER AS order_id,
       item.value:sku::VARCHAR AS sku,
       item.value:qty::INTEGER AS qty
FROM @LAB_FILES/json/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_JSON_ORDERS',
   PATTERN => '.*orders[.]json$') t,
LATERAL FLATTEN(INPUT => t.$1:items) item
ORDER BY order_id;
```

Expected: one `BOOK` item with quantity `1` for each order. `FLATTEN` produces one row per array element.

Source: [FLATTEN](https://docs.snowflake.com/en/sql-reference/functions/flatten).

## 14. XML example: source file

Save this as `xml/orders.xml` under the sample-data root, or use the included file. Upload it using section 7.

```xml
<?xml version='1.0' encoding='utf-8'?>
<orders>
  <order id="101">
    <customer>Asha</customer>
    <city>Chennai</city>
    <amount>1250.5</amount>
  </order>
  <order id="102">
    <customer>Ravi</customer>
    <city>Bengaluru</city>
    <amount>800.0</amount>
  </order>
  <order id="103">
    <customer>Meena</customer>
    <city>Hyderabad</city>
    <amount>1499.5</amount>
  </order>
</orders>
```

## XML: inspect documents, attributes, and element content

```sql
SELECT t.$1 AS parsed_xml
FROM @LAB_FILES/xml/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_XML_ORDERS',
   PATTERN => '.*orders[.]xml$') t;

SELECT GET(t.$1, '@id')::INTEGER AS order_id,
       GET(XMLGET(t.$1, 'customer'), '$')::VARCHAR AS customer,
       GET(XMLGET(t.$1, 'city'), '$')::VARCHAR AS city,
       GET(XMLGET(t.$1, 'amount'), '$')::NUMBER(10,2) AS amount
FROM @LAB_FILES/xml/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_XML_ORDERS',
   PATTERN => '.*orders[.]xml$') t
ORDER BY order_id;
```

Expected: the three orders from section 3.

`STRIP_OUTER_ELEMENT = TRUE` removes the enclosing `<orders>` wrapper, so each `$1` is an `<order>` representation. `GET(..., '@id')` reads its attribute. `XMLGET` returns a child element object; `GET(..., '$')` reads that element's content. It does not directly return text from `XMLGET` alone.

XML attributes, elements, and repeated tags need different navigation. The top-level order is already `$1`; asking `XMLGET($1, 'orders')` would search for a child that is no longer present. Cast the extracted text to the required business type.

Source: [XMLGET and XML representation](https://docs.snowflake.com/en/sql-reference/functions/xmlget).

## 15. Download a file with GET

Create a **local** destination folder in PowerShell:

```powershell
New-Item -ItemType Directory -Force -Path 'C:/Course/DataEng/D38_SnowFlake/downloads/csv'
```

Then run through a **local Snowflake SQL client**:

```sql
GET @STAGE_LAB_DB.PUBLIC.LAB_FILES/csv/
  'file://C:/Course/DataEng/D38_SnowFlake/downloads/csv/'
  PATTERN = '.*orders[.]csv$';
```

`GET` downloads from an internal stage to the client machine. It does not query records or remove the source. Download one format prefix at a time to avoid same-name collisions. `GET` does not automatically decompress gzip files. This lab uploaded without automatic compression.

Source: [GET](https://docs.snowflake.com/en/sql-reference/sql/get).

To view the downloaded CSV in PowerShell:

```powershell
Get-Content -LiteralPath 'C:/Course/DataEng/D38_SnowFlake/downloads/csv/orders.csv'
```

For an external stage, use an authorized cloud storage client to download the external object; SQL `GET` targets internal stages.

## 16. Avro example: upload and query a real binary file

The generator uses this schema with the same three order dictionaries. The `.avsc` file is schema text; `orders.avro` is the binary object container to upload.

```json
{
  "type": "record",
  "name": "Order",
  "namespace": "training",
  "fields": [
    {
      "name": "order_id",
      "type": "long"
    },
    {
      "name": "customer",
      "type": "string"
    },
    {
      "name": "city",
      "type": "string"
    },
    {
      "name": "amount",
      "type": "double"
    }
  ]
}
```

**Local Snowflake client:**

```sql
PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/avro/orders.avro'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/avro/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;
```

**Snowflake SQL:**

```sql
SELECT t.$1:order_id::INTEGER AS order_id,
       t.$1:customer::VARCHAR AS customer,
       t.$1:city::VARCHAR AS city,
       t.$1:amount::NUMBER(10,2) AS amount
FROM @LAB_FILES/avro/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_AVRO_ORDERS',
   PATTERN => '.*orders[.]avro$') t
ORDER BY order_id;
```

Expected: three orders. The reader decodes Avro records; it is not reading printable CSV lines. The teaching schema uses a double for simplicity, and SQL casts it to a decimal output.

Sources: [Avro writer](https://fastavro.readthedocs.io/en/latest/writer.html), [File formats](https://docs.snowflake.com/en/sql-reference/sql/create-file-format).

## 17. Parquet example: upload and query columns

The generator writes the three dictionaries as a PyArrow table to a Parquet file with Snappy compression inside the format. `AUTO_COMPRESS = FALSE` avoids adding another outer gzip layer; it does not remove Parquet's internal compression.

**Local Snowflake client:**

```sql
PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/parquet/orders.parquet'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/parquet/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;
```

**Snowflake SQL:**

```sql
SELECT t.$1:order_id::INTEGER AS order_id,
       t.$1:customer::VARCHAR AS customer,
       t.$1:city::VARCHAR AS city,
       t.$1:amount::NUMBER(10,2) AS amount
FROM @LAB_FILES/parquet/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_PARQUET_ORDERS',
   PATTERN => '.*orders[.]parquet$') t
ORDER BY order_id;
```

Expected: the same three rows. Parquet stores data in a binary columnar layout with schema metadata. You need a Parquet reader to inspect it locally; a text editor is not a useful way to read its business records.

Source: [Apache Arrow: Parquet](https://arrow.apache.org/docs/python/parquet.html).

## 18. Raw file versus parsed data

**Raw file** means the source bytes before business transformations. `SELECT $1` with a JSON or XML format gives a parsed value, not a byte-for-byte copy. Use `GET` to retrieve the original staged file.

For a plain text preview, upload the supplied text file through the local client:

```sql
PUT 'file://C:/Course/DataEng/D38_SnowFlake/data/stage_practical/raw/notes.txt'
  @STAGE_LAB_DB.PUBLIC.LAB_FILES/raw/
  AUTO_COMPRESS = FALSE OVERWRITE = FALSE;
```

Then query in Snowflake:

```sql
SELECT METADATA$FILE_ROW_NUMBER AS line_number, t.$1::VARCHAR AS raw_line
FROM @LAB_FILES/raw/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_RAW_LINES',
   PATTERN => '.*notes[.]txt$') t
ORDER BY line_number;
```

Expected lines: `Stage practical sample`, `Batch: D381`, `Expected orders: 3`.

The same raw reader can preview CSV lines including the header:

```sql
SELECT METADATA$FILE_ROW_NUMBER AS line_number, t.$1::VARCHAR AS raw_line
FROM @LAB_FILES/csv/
  (FILE_FORMAT => 'STAGE_LAB_DB.PUBLIC.FF_RAW_LINES',
   PATTERN => '.*orders[.]csv$') t
ORDER BY line_number;
```

Expected: four lines. This line-oriented view still parses records and line endings; it is not a general binary reader. Do not apply it to Avro or Parquet.

Source: [File-format options](https://docs.snowflake.com/en/sql-reference/sql/create-file-format).

## 19. Load staged CSV into a table

This optional step shows the difference between querying a stage and persisting typed rows.

```sql
CREATE TABLE IF NOT EXISTS ORDERS_CSV (
  order_id INTEGER,
  customer VARCHAR,
  city VARCHAR,
  amount NUMBER(10,2)
);

COPY INTO ORDERS_CSV
FROM @LAB_FILES/csv/
FILES = ('orders.csv')
FILE_FORMAT = (FORMAT_NAME = 'STAGE_LAB_DB.PUBLIC.FF_CSV_ORDERS')
ON_ERROR = 'ABORT_STATEMENT'
PURGE = FALSE;

SELECT COUNT(*) AS order_count, SUM(amount) AS total_amount FROM ORDERS_CSV;
LIST @LAB_FILES/csv/;
```

For a fresh table, expect three rows and total `3550.00`; the file remains staged. Review the `COPY INTO` status output. Snowflake tracks load history and ordinarily skips unchanged files already loaded into that table within its load-history rules. `FORCE = TRUE` can reload data and introduce duplicates, so it is not used here.

Source: [COPY INTO table](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table).

## 20. External stage: where the file physically lives

This is an optional template, not required for the local-file lab. Replace the names with a real bucket and an administrator-configured storage integration.

```sql
CREATE STAGE ORDERS_S3_STAGE
  URL = 's3://your-training-bucket/incoming/'
  STORAGE_INTEGRATION = YOUR_APPROVED_S3_INTEGRATION;

LIST @ORDERS_S3_STAGE/csv/;
```

Here `@ORDERS_S3_STAGE/csv/orders.csv` resolves to the object `s3://your-training-bucket/incoming/csv/orders.csv`. Upload to that bucket using the cloud provider's tools. SQL `PUT` uploads local files to internal stages, not external stages.

A stage definition alone cannot grant cloud access; the integration and cloud-side permissions must allow the path. Existing file formats can be used to query an authorized external stage with the same field-selection syntax as the internal examples.

Sources: [CREATE STAGE](https://docs.snowflake.com/en/sql-reference/sql/create-stage), [PUT](https://docs.snowflake.com/en/sql-reference/sql/put).

## 21. Troubleshooting the practical

| Symptom | Check or fix |
|---|---|
| `PUT` cannot find a local file | Run from a local client; verify the absolute file URI |
| Query returns no rows | Run `LIST`; verify prefix, case, suffix, and regex |
| Existing file changes are not visible | Check PUT status; deliberately re-upload that file with overwrite enabled |
| CSV header causes numeric conversion failure | Verify `SKIP_HEADER = 1` |
| JSON appears as one array | Verify `STRIP_OUTER_ARRAY = TRUE` |
| JSON field is NULL | Inspect `$1` and match the field's exact case |
| XML child lookup is NULL | Inspect `$1`; root is stripped; use `@id` for attributes |
| Parser reports unexpected content | Restrict path/pattern to one format; exclude `.avsc` |
| Directory query misses new files | Enable directory metadata and run `ALTER STAGE ... REFRESH` |
| No active warehouse | Run `USE WAREHOUSE STAGE_LAB_WH` |
| Object not found | Set database/schema or use fully qualified names |
| Permission denied | Check role privileges on database/schema, stage, formats, and warehouse |

For a named internal stage, `READ` allows reading/listing files and `WRITE` allows upload/removal; grant `READ` before `WRITE`. External stages use `USAGE` plus the appropriate integration/cloud access. Use an authorized role; broad administrator access is not a prerequisite for routine file queries.

Sources: [PUT access requirements](https://docs.snowflake.com/en/sql-reference/sql/put), [LIST access requirements](https://docs.snowflake.com/en/sql-reference/sql/list), [Directory management](https://docs.snowflake.com/en/user-guide/data-load-dirtables-manage).

## 22. Finish and verify

After all uploads:

```sql
LIST @LAB_FILES;
ALTER STAGE LAB_FILES REFRESH;
SELECT COUNT(*) AS staged_file_count FROM DIRECTORY(@LAB_FILES);
```

Expected: **6 files** if only the six sample data files were uploaded once under the specified prefixes. Every structured example should return the same three business orders. The raw text has three lines; CSV has four raw lines including its header.

Suspend the lab warehouse when finished and no lab query is running:

```sql
ALTER WAREHOUSE STAGE_LAB_WH SUSPEND;
```

If already suspended, Snowflake may report that state. Files remain stored; storage charges are independent of warehouse suspension. Removing files or dropping an internal stage deletes its stored files, so cleanup is a separate deliberate action after you no longer need the samples.

**Practice questions:**

1. Why can you list a staged CSV before creating a table?
2. Why does `DIRECTORY(@LAB_FILES)` return six rows while the CSV data query returns three?
3. What changes when `STRIP_OUTER_ARRAY` is false?
4. Why must a Parquet file be generated rather than renamed from CSV?
5. Which operation retrieves a file to your computer, and which loads rows into a table?

**Answers:** stages hold independent files; the queries count different things; the JSON outer array remains a single value; Parquet has a binary format and schema; `GET` downloads while `COPY INTO <table>` loads.